<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB14_Case_Study_ECMWF_Predicting_Wave_Height_from_Wind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **NB14 · Class 14 — Case Study: ECMWF Weather Data, Predicting Wave Height from Wind**

## Block 4: Proyectos — Case Studies (opening)

Blocks 2–3 taught the toolkit; Block 4 applies it to complete real case studies — closer to how you'll actually use this material after the course. This first case study uses **real ECMWF reanalysis data**, retrieved live from the same Copernicus Climate Data Store professionals use operationally, and asks a genuine question: **can wind alone predict wave height?** You will choose the modeling approach yourself, using `NB13`'s decision framework — this class does not tell you which architecture to use.

> **Important — do this before class**: register for a free Copernicus CDS account and generate your API key at [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) (an institutional email is recommended). Registration/approval can take a little time, so do this **before** the session, not during it.

### Learning objectives

By the end of this class, students will be able to:
- Retrieve real reanalysis data from the Copernicus Climate Data Store via the `cdsapi`.
- Explain what ERA5 reanalysis is and why it differs from a raw observation or a forecast.
- Reshape gridded NetCDF climate data into a flat, ML-ready table.
- Apply `NB13`'s decision framework to a new problem and justify a modeling choice.
- Train, compare, and evaluate models on a genuinely new real dataset, including a spatial sanity check.

### Agenda (2-hour class)

| # | Class segment | Approx. duration | Type |
|---|---------------------|:---:|:---:|
| 1 | Recap, Block 4 introduction, today's roadmap | 10 min | Theory |
| 2 | What is ECMWF/ERA5 reanalysis data? | 5 min | Theory |
| 3 | Setting up Copernicus CDS API access | 10 min | Practice |
| 4 | Downloading real ERA5 data | 10 min | Practice |
| 5 | Exploring the NetCDF grid | 15 min | Practice |
| 6 | Reshaping the grid into a flat table | 15 min | Practice |
| 7 | Exploring the real, flattened dataset | 10 min | Practice |
| 8 | Applying `NB13`'s decision framework | 10 min | Theory + Practice |
| 9 | Hands-on: training and comparing models | 25 min | Practice |
| 10 | Evaluation, interpretation, and what's next in Block 4 | 10 min | Practice |

> Timings are approximate guidance, not a strict script — there are no scheduled breaks. If we cover everything with time to spare, class ends early; that can happen and is fine.


---

## 1. Recap and Block 4 introduction

- **Block 2** (`NB02`–`NB06`): the classical ML toolkit.
- **Block 3** (`NB07`–`NB13`): Deep Learning, closing with a decision framework for choosing between everything learned so far.
- **Block 4** (starting today): complete case studies, applying that toolkit to new real problems.

Block 4 is also where the course's evaluation catches up with the teaching: your **individual final project** (40% of the grade — an unseen case study you present and defend) uses a *different* real dataset from the ones taught in class, and the **case-study submission** (20%) lets you pick and resubmit your own resolution of *any* notebook worked in class, including this one. From here on, some class sessions will be new case studies like this one, and others will be supervised time for you to work on your own project, with a short checkpoint due every session — ask your instructor for the exact schedule.

---

## 2. What is ECMWF/ERA5 reanalysis data?

The **[European Centre for Medium-Range Weather Forecasts (ECMWF)](https://en.wikipedia.org/wiki/ECMWF)** is an intergovernmental organization and one of the world's leading centers for numerical weather prediction. Its **[ERA5](https://en.wikipedia.org/wiki/ERA5)** dataset is a *reanalysis*: not a raw observation and not a forecast, but a physically consistent reconstruction of past atmospheric and ocean-surface conditions, produced by combining millions of real historical observations with a numerical weather model. In practice, this means ERA5 gives us **realistic, physically consistent global weather and sea-state data for any place and time since 1940** — exactly the kind of real environmental data a naval or ocean engineer would use for route planning, structural load estimates, or historical weather analysis.

---

## 3. Setting up Copernicus CDS API access

With your free API key from [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) ready, install the client and save your credentials for this session:

In [ ]:
%pip install -q cdsapi xarray netCDF4 cartopy

Run the cell below and paste your key **when prompted** — it uses `getpass`, which hides what you type/paste and, more importantly, means your real key is **never written into this notebook's code or saved output**. Unlike a hardcoded key sitting in a cell (which is exactly how this repository leaked a real API key earlier in this course's history — see the very first security fixes in this project), a `getpass` prompt only exists in the running session's memory; nothing about it ends up in this file when you save it, share it, or push it to GitHub.

1. Go to [cds.climate.copernicus.eu/profile](https://cds.climate.copernicus.eu/profile) and log in.
2. Copy your **Personal Access Token** (a long string of letters, numbers, and dashes).
3. Run the cell below. A hidden input box appears — paste your token there and press Enter.

In [ ]:
import os
from getpass import getpass

CDS_API_KEY = getpass("Paste your CDS API key from https://cds.climate.copernicus.eu/profile (input hidden): ").strip()

if not CDS_API_KEY:
    raise ValueError(
        "No key entered. Get one from https://cds.climate.copernicus.eu/profile "
        "and re-run this cell."
    )

cdsapirc = f"url: https://cds.climate.copernicus.eu/api\nkey: {CDS_API_KEY}\n"

with open(os.path.expanduser("~/.cdsapirc"), "w") as f:
    f.write(cdsapirc)

print("CDS API key saved for this session (not stored anywhere in this notebook).")

---

## 4. Downloading real ERA5 data

Request one real snapshot: 10 m wind components and significant wave height, over the North Atlantic/Western European shelf — a real, naval-relevant region (English Channel, Bay of Biscay, North Sea), on a single date and time. This keeps the download small while still real data, not simulated:

In [ ]:
import cdsapi

client = cdsapi.Client()

client.retrieve(
    "reanalysis-era5-single-levels",
    {
        "product_type": "reanalysis",
        "format": "netcdf",
        "variable": [
            "10m_u_component_of_wind",
            "10m_v_component_of_wind",
            "significant_height_of_combined_wind_waves_and_swell",
        ],
        "year": "2024",
        "month": "01",
        "day": "15",
        "time": ["12:00"],
        "area": [60, -20, 35, 10],  # North, West, South, East
    },
    "era5_snapshot.nc",
)

The Climate Data Store sometimes returns a single NetCDF file and sometimes a zip archive splitting variables by internal data "stream" — handle both cases so the rest of the notebook doesn't depend on which one you got:

In [ ]:
import zipfile
import glob

target = "era5_snapshot.nc"
extract_dir = "."

if zipfile.is_zipfile(target):
    extract_dir = "era5_extracted"
    with zipfile.ZipFile(target) as zf:
        zf.extractall(extract_dir)
    print("Zip archive detected and extracted:", os.listdir(extract_dir))
else:
    print("Single NetCDF file, no extraction needed.")

nc_files = glob.glob(os.path.join(extract_dir, "*.nc")) or [target]
print("NetCDF files:", nc_files)

---

## 5. Exploring the NetCDF grid

Open every file found and identify which one holds the wave variable (`swh`) and which holds the wind components (`u10`/`v10`) — this also protects the notebook against the file(s) coming back in a different arrangement than expected:

In [ ]:
import xarray as xr

datasets = [xr.open_dataset(f) for f in nc_files]
for i, ds in enumerate(datasets):
    print(f"File {i}: variables = {list(ds.data_vars)}, dims = {dict(ds.sizes)}")

wave_ds = next(ds for ds in datasets if "swh" in ds.data_vars)
wind_ds = next(ds for ds in datasets if "u10" in ds.data_vars and "v10" in ds.data_vars)

Plot the raw wave-height grid, the same first sanity check any real gridded dataset deserves before modeling anything:

In [ ]:
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

swh = wave_ds["swh"].isel(valid_time=0)
lon_min, lon_max = float(swh.longitude.min()), float(swh.longitude.max())
lat_min, lat_max = float(swh.latitude.min()), float(swh.latitude.max())

fig = plt.figure(figsize=(10, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

im = ax.imshow(swh.values, origin="upper", extent=[lon_min, lon_max, lat_min, lat_max],
                cmap="viridis", transform=ccrs.PlateCarree())

ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)

plt.colorbar(im, ax=ax, orientation="vertical", pad=0.05, shrink=0.7, label="Significant wave height (m)")
plt.title("Real ERA5 significant wave height snapshot")
plt.tight_layout()
plt.show()

With real coastlines and country borders drawn in, the blank/NaN region visibly lines up with land (France, the UK, Spain, Portugal) — wave height is only defined over open water, which becomes relevant in a moment. Being able to see *where* on Earth this data actually is matters for a naval/ocean case study, not just as decoration: it's how you'd recognize a fetch-limited enclosed sea (the English Channel, the North Sea) versus open Atlantic swell later in Part 7.

---

## 6. Reshaping the grid into a flat table

Every model we've used since `NB02` expects a flat table: one row per example, one column per feature. A spatial grid needs reshaping first — every `(latitude, longitude)` cell becomes one row.

One real wrinkle: ERA5's wave parameters (like `swh`) and its surface/atmospheric parameters (like `u10`/`v10`) are not always delivered on identically-shaped grids, even when requested together over the same area — a known quirk of how ECMWF's forecasting system represents ocean waves internally. Check both datasets' grid sizes first:

In [ ]:
print("wave_ds grid:", wave_ds.sizes)
print("wind_ds grid:", wind_ds.sizes)

If the two sizes above differ, the wind field needs interpolating onto the wave field's grid before they can share one table — done automatically below regardless of whether they matched or not, so this notebook works either way:

In [ ]:
import numpy as np
import pandas as pd

# Align the wind field onto the wave field's exact grid, whether or not they
# originally matched -- interpolation is a no-op if the grids already agree.
wind_ds_aligned = wind_ds.interp(latitude=wave_ds.latitude, longitude=wave_ds.longitude)

u10 = wind_ds_aligned["u10"].isel(valid_time=0).values
v10 = wind_ds_aligned["v10"].isel(valid_time=0).values
swh_vals = swh.values
lat_grid, lon_grid = np.meshgrid(wave_ds.latitude.values, wave_ds.longitude.values, indexing="ij")

assert u10.shape == swh_vals.shape == lat_grid.shape, (
    f"Grid shapes still don't match: u10 {u10.shape}, swh {swh_vals.shape}, "
    f"lat_grid {lat_grid.shape} -- inspect wave_ds/wind_ds.sizes above."
)

wind_speed = np.sqrt(u10 ** 2 + v10 ** 2)
wind_direction = (np.degrees(np.arctan2(u10, v10)) + 360) % 360

grid_df = pd.DataFrame({
    "latitude": lat_grid.ravel(),
    "longitude": lon_grid.ravel(),
    "u10": u10.ravel(),
    "v10": v10.ravel(),
    "wind_speed": wind_speed.ravel(),
    "wind_direction": wind_direction.ravel(),
    "swh": swh_vals.ravel(),
})

grid_df = grid_df.dropna()
print(grid_df.shape)
grid_df.head()

`dropna()` removed every land grid cell (no wave height defined there) in one step — a real, physically meaningful reason for missing data, different in kind from `NB09`'s sensor-file mismatch but handled the same way: understand *why* it's missing before deciding what to do about it.

---

## 7. Exploring the real, flattened dataset

`NB02`'s three starting questions, once more, on genuinely new real data:

In [ ]:
grid_df.describe()

And the relationship the whole class hinges on — does wave height actually track wind speed in this real data?

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(grid_df["wind_speed"], grid_df["swh"], alpha=0.3, s=10)
plt.xlabel("Wind speed (m/s)")
plt.ylabel("Significant wave height (m)")
plt.title("Wind speed vs. wave height, every real grid cell")
plt.show()

**Read your own plot**: is the relationship a clean line, a noisy trend, or close to no relationship at all? Real fetch-limited seas (an enclosed sea like parts of the English Channel, where wind doesn't have room to build up large waves) can look quite different from open Atlantic swell in the same snapshot — `latitude`/`longitude` may end up mattering as much as wind speed itself.

---

## 8. Applying `NB13`'s decision framework

Before writing any model code, run this real problem through `NB13`'s three questions:

1. **Labels?** Yes — `swh` is a real, known target for every grid cell.
2. **Data shape?** Tabular — each row is a single grid cell's features, not an image or a sequence (even though it originated from a spatial grid, we've already flattened it).
3. **Data volume?** Likely several thousand ocean grid cells after `dropna()` — moderate, not huge.

`NB13`'s framework, applied honestly, points toward **classical ML** as at least a strong baseline here — plausibly the right final answer too, exactly as it was for `NB06`'s yacht data in `NB13`'s own experiment. A neural network remains a legitimate choice to *also* try and compare, but the framework gives no reason to assume it will automatically win. **Your task**: pick at least one classical model and justify your choice out loud (to a classmate or your instructor) before writing the training code below.

---

## 9. Hands-on: training and comparing models

A leakage check first — `wind_speed` and `wind_direction` were both *derived* from `u10`/`v10`, so using all four together would just be feeding the model the same information twice in different forms, the same redundancy principle from `NB03`'s `CO2_emissions` example. Use `u10`/`v10` **or** `wind_speed`/`wind_direction`, not both representations at once:

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

feature_cols = ["latitude", "longitude", "wind_speed", "wind_direction"]
X = grid_df[feature_cols]
y = grid_df["swh"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

> **A real limitation worth naming, not hiding**: this is a *random* split of spatially correlated data — nearby grid cells have similar wave heights, so some information about the test cells likely leaks in through their train-set neighbors. A stricter evaluation would hold out an entire spatial region instead (try this as homework). We proceed with the random split for today, honestly labeled as a simplification, not a hidden flaw.

Compare a linear baseline against a tree ensemble — the classical candidates `NB13`'s framework favored:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score

candidate_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}
for name, model in candidate_models.items():
    scores = cross_val_score(model, X_train_scaled, y_train, cv=cv, scoring="r2")
    cv_results[name] = scores

pd.DataFrame(cv_results).mean().sort_values(ascending=False)

---

## 10. Evaluation, interpretation, and what's next in Block 4

Fit the stronger candidate on the full training set and evaluate once on the untouched test set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

best_model = RandomForestRegressor(n_estimators=200, random_state=42)
best_model.fit(X_train_scaled, y_train)
y_pred = best_model.predict(X_test_scaled)

print(f"MAE:  {mean_absolute_error(y_test, y_pred):.3f} m")
print(f"RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.3f} m")
print(f"R2:   {r2_score(y_test, y_pred):.3f}")

A spatial sanity check — plot the test-set residuals back on the map. Errors scattered randomly suggest a reasonably unbiased model; errors clustered in one region (say, the English Channel specifically) would suggest the model is systematically missing something about that area's sea state:

In [ ]:
residuals = y_test.values - y_pred

fig = plt.figure(figsize=(9, 6))
ax = plt.axes(projection=ccrs.PlateCarree())

sc = ax.scatter(X_test["longitude"], X_test["latitude"], c=residuals, cmap="coolwarm",
                 vmin=-abs(residuals).max(), vmax=abs(residuals).max(), s=15,
                 transform=ccrs.PlateCarree())

ax.coastlines(resolution="110m")
ax.add_feature(cfeature.BORDERS, linestyle=":")
ax.gridlines(draw_labels=True)

plt.colorbar(sc, ax=ax, orientation="vertical", pad=0.05, shrink=0.7, label="Residual (actual - predicted), m")
plt.title("Where does the model's error concentrate?")
plt.tight_layout()
plt.show()

---

### What's next in Block 4

The next taught sessions in this block cover other real case studies (historical voyage records, terrain/elevation data). The sessions *between* them are your supervised project time — bring real, checkpointable progress on your individual final project every time, per the schedule in `Final_Project_Wave_Height_Forecasting_STARTER.ipynb`'s rubric (note: that project uses a **different** real dataset from today's, by design — see that notebook's rules).

---

## Class summary

- ERA5 reanalysis provides real, physically consistent historical weather/ocean data, retrieved live via the Copernicus CDS API — a genuine professional workflow, not a teaching shortcut.
- Gridded NetCDF data reshapes into a flat table exactly like any other dataset once you understand its dimensions — `dropna()` removing land cells was a physically meaningful cleaning step, not an arbitrary one.
- `NB13`'s decision framework, applied to a genuinely new problem, pointed toward classical ML — and a quick cross-validated comparison backed that up.
- A random train/test split on spatial data has a real leakage caveat worth naming honestly, not hiding.
- A residual map is a spatial-data-specific interpretation tool, alongside the usual MAE/RMSE/R².

## Homework / Practice Ideas

1. Change the `area` in Part 4 to a different real region (e.g., the Mediterranean, or waters near your own country) and re-run the notebook — does the wind-vs-wave relationship from Part 7 look similar?
2. Implement the stricter spatial holdout suggested in Part 9: split by a longitude threshold (e.g., train on everything west of -5°, test on everything east of it) instead of a random split — how much does the reported R² change?
3. Add a small MLP (`NB07` style) to Part 9's comparison — does it beat the Random Forest here, and does that match or contradict `NB13`'s general expectation?
4. Request a second date (e.g., a stormy day vs. a calm day) and compare the two wind-vs-wave relationships from Part 7 — is the relationship stronger under storm conditions?
5. Using the residual map from Part 10, identify the single worst-predicted region and propose (in a markdown cell, no code needed) a feature that might explain that error.

> ***As always: a new case study is only "solved" once you can explain both what the model got right and where it struggled — the residual map in Part 10 is not optional decoration.***
